In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Đường dẫn đến dữ liệu đã được chia sẵn
data_train_path = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_split_03[xóa z và visibility]/train"
data_val_path   = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_split_03[xóa z và visibility]/val"
data_test_path  = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_split_03[xóa z và visibility]/test"

# Siêu tham số
learning_rate = 1e-4
epochs = 200
patience = 30
mask_value = 0
loss='sparse_categorical_crossentropy'

# Các giá trị thử nghiệm cho batch_size, sequence_length và overlap
batch_sizes = [4]
sequence_lengths = [20]
overlap_values = [0.5]
num_joints = 15

# Hàm load dữ liệu (mỗi file CSV chứa dữ liệu của 1 sample)
def load_data_with_fixed_length(base_path, sequence_length, overlap):
    X, y = [], []
    # Lấy danh sách các thư mục con (mỗi thư mục là 1 hành động)
    actions = sorted(os.listdir(base_path))
    for label, action in enumerate(actions):
        action_path = os.path.join(base_path, action)
        if os.path.isdir(action_path):
            for file in os.listdir(action_path):
                file_path = os.path.join(action_path, file)
                if file.endswith('.csv'):
                    # Bỏ cột đầu tiên nếu không cần thiết
                    df = pd.read_csv(file_path).iloc[:, 1:]
                    total_rows = df.shape[0]
                    # Cắt đoạn dữ liệu với bước nhảy: (sequence_length - overlap)
                    for start_idx in range(0, total_rows, sequence_length - overlap):
                        end_idx = start_idx + sequence_length
                        if end_idx > total_rows:
                            break
                        segment = df.iloc[start_idx:end_idx].values
                        X.append(segment)
                        y.append(label)
    return np.array(X), np.array(y)

# Định nghĩa lớp ST-GCN block sử dụng định dạng NHWC (channels_last)
class STGCNBlock(tf.keras.layers.Layer):
    def __init__(self, out_channels, A, kernel_size=9, stride=1, use_residual=True, **kwargs):
        """
        out_channels: số kênh đầu ra của block
        A: ma trận kề (shape: [num_joints, num_joints])
        kernel_size: kích thước kernel cho phép biến đổi thời gian
        stride: bước nhảy trên trục thời gian
        """
        super(STGCNBlock, self).__init__(**kwargs)
        self.out_channels = out_channels
        self.A = A  # ma trận kề cố định
        self.kernel_size = kernel_size
        self.stride = stride
        self.use_residual = use_residual
        # Không cần chỉ định data_format, mặc định là channels_last
        self.conv = tf.keras.layers.Conv2D(
            out_channels,
            kernel_size=(kernel_size, 1),
            strides=(stride, 1),
            padding='same'
        )
        self.bn = tf.keras.layers.BatchNormalization(axis=-1)
        self.relu = tf.keras.layers.ReLU()
        self.residual_conv = None

    def build(self, input_shape):
        in_channels = input_shape[-1]  # với NHWC, channel nằm ở cuối
        if self.use_residual:
            if in_channels != self.out_channels or self.stride != 1:
                self.residual_conv = tf.keras.layers.Conv2D(
                    self.out_channels,
                    kernel_size=1,
                    strides=(self.stride, 1),
                    padding='same'
                )
                self.residual_bn = tf.keras.layers.BatchNormalization(axis=-1)
        super(STGCNBlock, self).build(input_shape)

    def call(self, x):
        # x có shape: (N, T, V, C)
        res = x
        # Phép biến đổi không gian: nhân theo ma trận kề trên trục các khớp (V)
        # Ta nhân theo trục V: 'ntvc, vw -> ntwc'
        x = tf.einsum('ntvc,vw->ntwc', x, self.A)
        x = self.conv(x)
        x = self.bn(x)
        if self.use_residual:
            if self.residual_conv is not None:
                res = self.residual_conv(res)
                res = self.residual_bn(res)
            x = x + res
        return self.relu(x)

# Hàm xây dựng mô hình ST-GCN với định dạng NHWC
def build_stgcn_model(num_joints, in_channels, sequence_length, num_classes):
    """
    Input shape: (sequence_length, num_joints, in_channels) với định dạng NHWC
    """
    inputs = tf.keras.Input(shape=(sequence_length, num_joints, in_channels))
    # Sử dụng ma trận kề đơn giản: ma trận đơn vị (self-connection)
    A = tf.constant(np.eye(num_joints), dtype=tf.float32)

    # Xây dựng 3 block ST-GCN liên tiếp
    x = STGCNBlock(64, A, kernel_size=9, stride=1)(inputs)
    x = STGCNBlock(128, A, kernel_size=9, stride=1)(x)
    x = STGCNBlock(256, A, kernel_size=9, stride=1)(x)

    # Global Average Pooling trên trục thời gian và các khớp
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

# Vòng lặp thử nghiệm với các giá trị batch_size, sequence_length và overlap khác nhau
for batch_size in batch_sizes:
    for seq_len in sequence_lengths:
        for overlap_ratio in overlap_values:
            # Tính overlap dưới dạng số bước (int)
            overlap = int(overlap_ratio * seq_len)

            print(f"\nTraining with batch_size={batch_size}, sequence_length={seq_len}, overlap={overlap}")

            # Load dữ liệu từ các folder đã chia sẵn
            X_train, y_train = load_data_with_fixed_length(data_train_path, seq_len, overlap)
            X_val, y_val     = load_data_with_fixed_length(data_val_path, seq_len, 0)
            X_test, y_test   = load_data_with_fixed_length(data_test_path, seq_len, 0)

            # Tiền xử lý: Scale dữ liệu sử dụng MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
            X_val   = scaler.transform(X_val.reshape(-1, 1)).reshape(X_val.shape)
            X_test  = scaler.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)

            # Xử lý NaN
            X_train = np.nan_to_num(X_train, nan=mask_value)
            X_val   = np.nan_to_num(X_val, nan=mask_value)
            X_test  = np.nan_to_num(X_test, nan=mask_value)

            # Sau khi load, dữ liệu có shape: (num_samples, seq_len, D)
            # Ta cần reshape thành (num_samples, seq_len, num_joints, in_channels)
            D = X_train.shape[2]
            if D % num_joints != 0:
                raise ValueError(f"Số cột ({D}) không chia hết cho num_joints ({num_joints}). Kiểm tra lại dữ liệu!")
            in_channels = D // num_joints

            # Reshape dữ liệu (không cần chuyển đổi sang channels_first)
            X_train = X_train.reshape(X_train.shape[0], seq_len, num_joints, in_channels)
            X_val   = X_val.reshape(X_val.shape[0], seq_len, num_joints, in_channels)
            X_test  = X_test.reshape(X_test.shape[0], seq_len, num_joints, in_channels)

            # Tính class weight để cân bằng các lớp
            class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
            class_weights = dict(enumerate(class_weights))

            # Xác định số lớp (num_classes)
            num_classes = len(np.unique(y_train))

            # Xây dựng mô hình ST-GCN
            model = build_stgcn_model(num_joints, in_channels, seq_len, num_classes)
            model.compile(optimizer=Adam(learning_rate=learning_rate),
                          loss=loss,
                          metrics=['accuracy'])

            early_stopping = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

            # Huấn luyện mô hình
            history = model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=batch_size,
                validation_data=(X_val, y_val),
                class_weight=class_weights,
                callbacks=[early_stopping],
                verbose=1
            )

            # Lưu mô hình
            model_name = f"stgcn_model_bs{batch_size}_sl{seq_len}_ol{overlap}.keras"
            save_path = os.path.join("/content/drive/MyDrive/Gym/Model", model_name)
            model.save(save_path)
            print(f"Model saved to {save_path}")

            # Đánh giá trên tập test
            test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
            print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Training with batch_size=4, sequence_length=20, overlap=10
Epoch 1/200
4906/4906 ━━━━━━━━━━━━━━━━━━━━ 412s 83ms/step - accuracy: 0.1372 - loss: 2.8143 - val_accuracy: 0.1750 - val_loss: 3.3156
Epoch 2/200
4906/4906 ━━━━━━━━━━━━━━━━━━━━ 458s 86ms/step - accuracy: 0.2270 - loss: 2.5145 - val_accuracy: 0.1396 - val_loss: 4.0486
Epoch 3/200
4906/4906 ━━━━━━━━━━━━━━━━━━━━ 435s 85ms/step - accuracy: 0.2593 - loss: 2.4198 - val_accuracy: 0.1523 - val_loss: 3.1556
Epoch 4/200
4906/4906 ━━━━━━━━━━━━━━━━━━━━ 440s 84ms/step - accuracy: 0.2907 - loss: 2.3495 - val_accuracy: 0.1424 - val_loss: 3.3485
Epoch 5/200
4906/4906 ━━━━━━━━━━━━━━━━━━━━ 442s 85ms/step - accuracy: 0.3042 - loss: 2.2885 - val_accuracy: 0.1250 - val_loss: 3.9669
Epoch 6/200
4906/4906 ━━━━━━━━━━━━━━━━━━━━ 428s 82ms/step - accuracy: 0.3078 - loss: 2.2587 - val_accuracy: 0.1653 - val_loss: 3.8205
Epoch 7